In [9]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate

In [10]:
densityfname = "example_density_phases_dump.csv"
data = np.genfromtxt(densityfname, delimiter=",", names=True)
time = data["time"]

In [11]:
# All frames phase-specific density
rho_all_dense = data["rho_dense_all"]
rho_all_dilute = data["rho_dilute_all"]
rho1_dense = data["rho_dense_Mol1"]
rho1_dilute = data["rho_dilute_Mol1"]
rho2_dense = data["rho_dense_Mol2"]
rho2_dilute = data["rho_dilute_Mol2"]

In [12]:
# All frames phase-specific volume fraction
def rho_to_phi(rho):
    return rho * np.pi / 6
phi_all_dense = rho_to_phi(rho_all_dense)
phi_all_dilute = rho_to_phi(rho_all_dilute)
phi1_dense = rho_to_phi(rho1_dense)
phi1_dilute = rho_to_phi(rho1_dilute)
phi2_dense = rho_to_phi(rho2_dense)
phi2_dilute = rho_to_phi(rho2_dilute)

In [13]:
# K2
K2 = np.mean(rho2_dense) / np.mean(rho2_dilute)

In [20]:
# Statistic calculations
def calc_autocorrelation(data, time=None):
    data = np.asarray(data)
    fluctuations = data - np.mean(data)
    acf_full = correlate(fluctuations, fluctuations, mode="full", method="fft")
    acf = acf_full[len(fluctuations)-1:]
    normalization = np.arange(len(data), 0, -1)
    acf = acf / normalization
    acf = acf / acf[0]
    if time is None:
        lag_times = np.arange(len(acf))
    else:
        time = np.asarray(time)
        if len(time) != len(data):
            raise ValueError("ERROR: Time and data must have the same length.")
        dt = np.mean(np.diff(time))
        lag_times = np.arange(len(acf))*dt
    return lag_times, acf

def calc_integrated_autocorrelation_time(lag_times, acf):
    lag_times, acf = np.asarray(lag_times), np.asarray(acf)
    nonpositive = np.where(acf <= 0)[0]
    if len(nonpositive) > 0:
        stop = nonpositive[0]
    else:
        stop = len(acf)
    dt = lag_times[1] - lag_times[0]
    tau_int = dt * (0.5 + np.sum(acf[1:stop]))
    return tau_int, stop

def calc_effective_sample_size(n_samples, lag_times, tau_int):
    dt = lag_times[1] - lag_times[0]
    n_eff = (n_samples * dt / (2*tau_int))
    return n_eff

def calc_block_sem(data, block_size):
    data = np.asarray(data)
    n_samples = len(data)
    n_blocks = n_samples // block_size
    n_used = n_blocks * block_size
    blocked_data = data[:n_used]
    blocks = blocked_data.reshape(n_blocks, block_size)
    block_means = np.mean(blocks, axis=1)
    sem = np.std(block_means, ddof=1) / np.sqrt(n_blocks)
    return block_means, sem

In [42]:
# Statistic parameter searches
def generate_block_sizes(n_samples, tau_int, dt, min_blocks=5, tau_factor=2, growth_factor=1.25):
    min_block_size = int(np.ceil(tau_factor * tau_int / dt))
    min_block_size = max(1, min_block_size)
    max_block_size = n_samples // min_blocks
    block_sizes = []
    block_size = min_block_size
    while block_size <= max_block_size:
        block_sizes.append(block_size)
        next_block_size = int(np.ceil(block_size * growth_factor))
        if next_block_size == block_size:
            next_block_size += 1
        block_size = next_block_size
    return np.asarray(block_sizes, dtype=int)

def scan_block_sem(data, block_sizes, min_blocks=5):
    data = np.asarray(data)
    valid_block_sizes, sem_values, n_blocks_values = [], [], []
    for block_size in block_sizes:
        n_blocks = len(data) // block_size
        if n_blocks < min_blocks:
            continue
        _, sem = calc_block_sem(data, block_size)
        valid_block_sizes.append(block_size)
        sem_values.append(sem)
        n_blocks_values.append(n_blocks)
    return (np.asarray(valid_block_sizes), np.asarray(sem_values), np.asarray(n_blocks_values))

def detect_sem_plateau(block_sizes, sem_values, n_blocks, tolerance=0.10, window=4, min_blocks=5):
    block_sizes, sem_values, n_blocks = np.asarray(block_sizes), np.asarray(sem_values), np.asarray(n_blocks)
    for i in range(len(block_sizes) - window + 1):
        sem_window = sem_values[i:i + window]
        n_blocks_window = n_blocks[i:i + window]
        if np.any(n_blocks_window < min_blocks):
            continue
        mean_sem = np.mean(sem_window)
        relative_spread = (np.max(sem_window) - np.min(sem_window)) / mean_sem
        if relative_spread <= tolerance:
            selected_index = i + window - 1
            return block_sizes[selected_index], sem_values[selected_index], selected_index
    return None, None, None

def calc_blocking_statistics(data, time, min_blocks=10, tau_factor=2, growth_factor=1.20, plateau_tolerance=0.10, plateau_window=6):
    data, time = np.asarray(data), np.asarray(time)
    lag_times, acf = calc_autocorrelation(data, time)
    tau_int, acf_stop = calc_integrated_autocorrelation_time(lag_times, acf)
    n_eff = calc_effective_sample_size(len(data), lag_times, tau_int)
    dt = np.mean(np.diff(time))
    mean = np.mean(data)
    std = np.std(data, ddof=1)
    sem_acf = std / np.sqrt(n_eff)
    n_samples = len(data)
    
    # Autocorrelation statistics
    lag_times, acf = calc_autocorrelation(data, time)
    tau_int, acf_stop = calc_integrated_autocorrelation_time(lag_times, acf)
    n_eff = calc_effective_sample_size(n_samples, lag_times, tau_int)

    # Candidate block sizes
    candidate_block_sizes = generate_block_sizes(n_samples=n_samples, tau_int=tau_int, dt=dt, min_blocks=min_blocks, tau_factor=tau_factor, growth_factor=growth_factor)

    block_sizes, sem_values, n_blocks = scan_block_sem(data, candidate_block_sizes, min_blocks=min_blocks)

    # Find SEM plateau
    plateau_block_size, plateau_sem, plateau_index = detect_sem_plateau(block_sizes, sem_values, n_blocks, tolerance=plateau_tolerance, window=plateau_window, min_blocks=min_blocks)

    if plateau_index is None:
        plateau_n_blocks = None
        plateau_n_used = None
        plateau_block_time = None
    else:
        plateau_n_blocks = int(n_blocks[plateau_index])
        plateau_n_used = plateau_n_blocks * plateau_block_size
        plateau_block_time = plateau_block_size * dt

    return {
        "mean": mean,
        "std": std,
        "n_samples": n_samples,
        "dt": dt,

        "tau_int": tau_int,
        "acf_stop": acf_stop,
        "n_eff": n_eff,

        "block_sizes": block_sizes,
        "block_times": block_sizes * dt,
        "sem_values": sem_values,
        "n_blocks": n_blocks,

        "sem": plateau_sem,
        "block_size": plateau_block_size,
        "block_time": plateau_block_time,
        "n_blocks_used": plateau_n_blocks,
        "n_samples_used": plateau_n_used,
        "sem_acf": std / np.sqrt(n_eff),

        "plateau_index": plateau_index,

        "min_blocks": min_blocks,
        "tau_factor": tau_factor,
        "growth_factor": growth_factor,
        "plateau_tolerance": plateau_tolerance,
        "plateau_window": plateau_window,

        "lag_times": lag_times,
        "acf": acf,
    }

In [45]:
# Calculate statistics
stats_rho_all_dense = calc_blocking_statistics(rho_all_dense, time)
stats_rho_all_dilute = calc_blocking_statistics(rho_all_dilute, time)
stats_rho1_dense = calc_blocking_statistics(rho1_dense, time)
stats_rho1_dilute = calc_blocking_statistics(rho1_dilute, time)
stats_rho2_dense = calc_blocking_statistics(rho2_dense, time)
stats_rho2_dilute = calc_blocking_statistics(rho2_dilute, time)

In [35]:
np.mean(rho_all_dense)

np.float64(0.9115336781029264)